# Semana 9 — Lista de Prática SQL para Data Warehouse

Este notebook contém **10 exercícios progressivos** para praticar os conteúdos da Semana 9 usando o mesmo modelo de dados trabalhado em aula.

## Conteúdos praticados

- `SELECT`, `WHERE`, `ORDER BY` e `LIMIT`
- `INNER JOIN` entre fato e dimensões
- `LEFT JOIN`
- `GROUP BY`, `SUM`, `COUNT` e `AVG`
- `WHERE` antes da agregação
- `HAVING` depois da agregação
- `DATE_TRUNC` e `EXTRACT`

## Tabelas usadas

- `fato_vendas`: tabela fato com as vendas
- `dim_cliente`: dimensão com cidade e estado do cliente
- `dim_produto`: dimensão com categoria do produto
- `dim_tempo`: dimensão com datas, ano, mês, trimestre e ano/mês

> Orientação: escreva sua resposta dentro da célula de código de cada exercício, substituindo o comentário `-- escreva sua consulta aqui`.


## 1. Preparação do ambiente

Execute as células abaixo antes de iniciar os exercícios. Os arquivos `.csv` devem estar na mesma pasta deste notebook. Caso estejam em outra pasta, altere o valor da variável `BASE_PATH`.


In [2]:
# Instalação, caso esteja usando Google Colab ou ambiente sem DuckDB
# !pip install duckdb pandas

In [15]:
import pandas as pd
import duckdb
from pathlib import Path

# Altere aqui se os arquivos estiverem em outra pasta
BASE_PATH = Path('../Dataset_schema')

dim_cliente = pd.read_csv(BASE_PATH / 'dim_cliente.csv')
dim_produto = pd.read_csv(BASE_PATH / 'dim_produto.csv')
dim_tempo = pd.read_csv(BASE_PATH / 'dim_tempo.csv')
fato_vendas = pd.read_csv(BASE_PATH / 'fato_vendas.csv')

banco_dados = duckdb.connect()
banco_dados.register('dim_cliente', dim_cliente)
banco_dados.register('dim_produto', dim_produto)
banco_dados.register('dim_tempo', dim_tempo)
banco_dados.register('fato_vendas', fato_vendas)

print('Tabelas carregadas com sucesso!')

Tabelas carregadas com sucesso!


## 2. Consulta rápida das tabelas

Use esta parte para lembrar quais colunas existem em cada tabela.


In [4]:
print('fato_vendas')
display(fato_vendas.head(1))

print('dim_cliente')
display(dim_cliente.head(1))

print('dim_produto')
display(dim_produto.head(1))

print('dim_tempo')
display(dim_tempo.head(1))

fato_vendas


,venda_sk,order_id,order_item_id,cliente_sk,produto_sk,tempo_sk,valor_produto,valor_frete,valor_total
0,1,00010242fe8c5a6d1ba2dd792cb16214,1,65558,25866,64,58.9,13.29,72.19


dim_cliente


,cliente_sk,customer_id,customer_unique_id,customer_city,customer_state
0,1,06b8999e2fba1a1fbc88172c00ba8bc7,861eff4711a542e4b93843c6dd7febb0,franca,SP


dim_produto


,produto_sk,product_id,product_category_name,product_category_name_english
0,1,1e9e8ef04dbcff4541ed26657ea517e5,perfumaria,perfumery


dim_tempo


,tempo_sk,data,ano,mes,dia,trimestre,ano_mes
0,1,2017-10-02,2017,10,2,4,2017-10


# 10 exercícios de prática

## Exercício 1 🟢 — Primeiras vendas da tabela fato

Liste as **20 primeiras vendas** da tabela `fato_vendas`.

Mostre as colunas:

- `venda_sk`
- `order_id`
- `valor_produto`
- `valor_frete`
- `valor_total`

**Objetivo:** praticar `SELECT`, `FROM` e `LIMIT`.

In [16]:
banco_dados.sql("""
    SELECT
        venda_sk,
        order_id,
        valor_produto,
        valor_frete,
        valor_total
    FROM fato_vendas
    LIMIT 20;
""").df()

,venda_sk,order_id,valor_produto,valor_frete,valor_total
0,1,00010242fe8c5a6d1ba2dd792cb16214,58.90,13.29,72.19
1,2,00018f77f2f0320c557190d7a144bdd3,239.90,19.93,259.83
2,3,000229ec398224ef6ca0657da4fc703e,199.00,17.87,216.87
3,4,00024acbcdf0a6daa1e931b038114c75,12.99,12.79,25.78
4,5,00042b26cf59d7ce69dfabb4e55b4fd9,199.90,18.14,218.04
5,6,00048cc3ae777c65dbb7d2a0634bc1ea,21.90,12.69,34.59
6,7,00054e8431b9d7675808bcb819fb4a32,19.90,11.85,31.75
7,8,000576fe39319847cbb9d288c5617fa6,810.00,70.75,880.75
8,9,0005a1a1728c9d785b8e2b08b904576c,145.95,11.65,157.60
9,10,0005f50442cb953dcd1d21e1fb923495,53.99,11.40,65.39


## Exercício 2 🟢 — Filtro com WHERE

Liste as vendas em que o `valor_total` seja **maior que 300**.

Mostre as colunas:

- `venda_sk`
- `order_id`
- `valor_total`

Ordene o resultado do **maior para o menor valor total**.

**Objetivo:** praticar `WHERE` e `ORDER BY DESC`.

In [6]:
banco_dados.sql("""
    SELECT
        venda_sk,
        order_id,
        valor_total
    FROM fato_vendas
    WHERE valor_total > 300    
    ORDER BY valor_total DESC;
""").df()

,venda_sk,order_id,valor_total
0,3482,0812eb902a67711a1cb742b3cdaa65ae,6929.31
1,109785,fefacc66af859508bf1a7934eab1e97f,6922.21
2,105496,f5136e38d1a14a4dbd87dff67da82701,6726.66
3,72721,a96610ab360d42a2e5335a3998b4718a,4950.34
4,10998,199af31afc78c699f0dbf71fb178d4d4,4764.34
...,...,...,...
8498,17744,299ad5e33a5f4fe1efcac35b99b0fad8,300.21
8499,105349,f4ae2ab52b8c6e093b06d6656ee4d351,300.12
8500,11827,1b6c00256bc49004b1d059039306c08e,300.12
8501,25863,3c38b53bb7d6f3c9f00d993a80f8cdd5,300.11


## Exercício 3 🟢 — Ranking de fretes

Liste as **15 vendas com maior valor de frete**.

Mostre as colunas:

- `venda_sk`
- `order_id`
- `valor_frete`
- `valor_total`

**Objetivo:** praticar ordenação e criação de ranking com `ORDER BY` e `LIMIT`.

In [7]:
banco_dados.sql("""
    SELECT
        venda_sk,
        order_id,
        valor_frete,
        valor_total
    FROM fato_vendas
    ORDER BY valor_frete DESC
    LIMIT 15;
""").df()

,venda_sk,order_id,valor_frete,valor_total
0,71885,a77e1550db865202c56b19ddc6dc4d53,409.68,1388.68
1,3232,076d1555fb53a89b0ef4d529e527a0f6,375.28,2713.36
2,27413,3fde74c28a3d5d618c00f26d51baafa0,375.28,2713.36
3,68275,9f49bd16053df810384e793386312674,339.59,1488.59
4,16360,264a7e199467906c0727394df82d1a6a,338.30,1388.30
5,86056,c7a07ddd52bbe18b61da49a8d89853d3,322.10,1372.10
6,4939,0b6230647ed16f4b3e70282dc4b5b87f,321.88,1371.88
7,3510,0822bcde10bb5d023755a71bc8f7797f,321.46,1311.46
8,29114,43bdbd9dc0931d72befdf4765af6c442,317.47,3406.47
9,47246,6ddfbf514959b49b6410c01ad93054bb,314.40,1359.40


## Exercício 4 🟡 — INNER JOIN com clientes

Mostre as vendas junto com o **estado e a cidade do cliente**.

Use `INNER JOIN` entre `fato_vendas` e `dim_cliente`.

Mostre as colunas:

- `f.venda_sk`
- `f.order_id`
- `c.customer_city`
- `c.customer_state`
- `f.valor_total`

Mostre apenas 20 linhas.

**Objetivo:** praticar relacionamento entre tabela fato e dimensão cliente.

In [8]:
banco_dados.sql("""
SELECT
    f.venda_sk,
    f.order_id,
    c.customer_city AS Cidade,
    c.customer_state AS Estado,
    f.valor_total
FROM fato_vendas f
INNER JOIN dim_cliente c
    ON f.cliente_sk = c.cliente_sk
LIMIT 20;
""").df()

,venda_sk,order_id,Cidade,Estado,valor_total
0,1,00010242fe8c5a6d1ba2dd792cb16214,campos dos goytacazes,RJ,72.19
1,2,00018f77f2f0320c557190d7a144bdd3,santa fe do sul,SP,259.83
2,3,000229ec398224ef6ca0657da4fc703e,para de minas,MG,216.87
3,4,00024acbcdf0a6daa1e931b038114c75,atibaia,SP,25.78
4,5,00042b26cf59d7ce69dfabb4e55b4fd9,varzea paulista,SP,218.04
5,6,00048cc3ae777c65dbb7d2a0634bc1ea,uberaba,MG,34.59
6,7,00054e8431b9d7675808bcb819fb4a32,guararapes,SP,31.75
7,8,000576fe39319847cbb9d288c5617fa6,praia grande,SP,880.75
8,9,0005a1a1728c9d785b8e2b08b904576c,santos,SP,157.60
9,10,0005f50442cb953dcd1d21e1fb923495,jandira,SP,65.39


## Exercício 5 🟡 — INNER JOIN com produto e tempo

Mostre as vendas com **categoria do produto** e **mês/ano da venda**.

Use `INNER JOIN` entre:

- `fato_vendas` e `dim_produto`
- `fato_vendas` e `dim_tempo`

Mostre as colunas:

- `f.order_id`
- `p.product_category_name_english AS categoria`
- `t.ano_mes`
- `f.valor_total`

Mostre apenas 30 linhas.

**Objetivo:** praticar mais de um `JOIN` na mesma consulta.

In [9]:
banco_dados.sql("""
SELECT
    f.order_id,
    p.product_category_name_english AS categoria,
    t.ano_mes,
    f.valor_total
FROM fato_vendas f
INNER JOIN dim_produto p
    ON f.produto_sk = p.produto_sk
INNER JOIN dim_tempo t
    ON f.tempo_sk = t.tempo_sk
LIMIT 30;
""").df()

,order_id,categoria,ano_mes,valor_total
0,00010242fe8c5a6d1ba2dd792cb16214,cool_stuff,2017-09,72.19
1,00018f77f2f0320c557190d7a144bdd3,pet_shop,2017-04,259.83
2,000229ec398224ef6ca0657da4fc703e,furniture_decor,2018-01,216.87
3,00024acbcdf0a6daa1e931b038114c75,perfumery,2018-08,25.78
4,00042b26cf59d7ce69dfabb4e55b4fd9,garden_tools,2017-02,218.04
5,00048cc3ae777c65dbb7d2a0634bc1ea,housewares,2017-05,34.59
6,00054e8431b9d7675808bcb819fb4a32,telephony,2017-12,31.75
7,000576fe39319847cbb9d288c5617fa6,garden_tools,2018-07,880.75
8,0005a1a1728c9d785b8e2b08b904576c,health_beauty,2018-03,157.60
9,0005f50442cb953dcd1d21e1fb923495,books_technical,2018-07,65.39


## Exercício 6 🟡 — Faturamento por estado

Calcule o **faturamento total por estado do cliente**.

Use `fato_vendas` e `dim_cliente`.

A consulta deve retornar:

- `estado`
- `qtd_pedidos`, contando pedidos únicos com `COUNT(DISTINCT f.order_id)`
- `faturamento_total`, usando `SUM(f.valor_total)`
- `ticket_medio`, usando `AVG(f.valor_total)`

Ordene pelo maior faturamento.

**Objetivo:** praticar `GROUP BY`, `SUM`, `COUNT`, `AVG` e `JOIN`.

In [10]:
banco_dados.sql("""
SELECT
    c.customer_state AS estado,
    COUNt(DISTINCT f.order_id) as qtd_pedidos,
    SUM(f.valor_total) AS faturamento_total,
    AVG(f.valor_total) as ticket_medio
FROM fato_vendas f
INNER JOIN dim_cliente c
    ON f.cliente_sk = c.cliente_sk
GROUP BY estado
ORDER BY faturamento_total DESC;
""").df()

,estado,qtd_pedidos,faturamento_total,ticket_medio
0,SP,40501,5769703.15,124.218549
1,RJ,12350,2055401.57,145.329956
2,MG,11354,1818891.67,140.824688
3,RS,5345,861472.79,140.442255
4,PR,4923,781708.80,138.380032
5,SC,3546,595127.78,145.259404
6,BA,3256,591137.81,160.504428
7,DF,2080,346123.35,146.973822
8,GO,1957,334212.35,146.777492
9,ES,1995,317657.93,142.767609


## Exercício 7 🟡 — Receita por categoria somente em SP

Calcule a **receita por categoria de produto** considerando apenas clientes do estado de **SP**.

Use `fato_vendas`, `dim_produto` e `dim_cliente`.

A consulta deve retornar:

- `categoria`
- `qtd_pedidos`
- `receita_total`

Ordene pela maior receita.

**Objetivo:** praticar `WHERE` antes do `GROUP BY`.

In [11]:
banco_dados.sql("""
SELECT
    p.product_category_name AS categoria,
    COUNt(DISTINCT f.order_id) as qtd_pedidos,
    SUM(f.valor_total) AS receita_total,
FROM fato_vendas f
INNER JOIN dim_produto p
    ON f.produto_sk = p.produto_sk
INNER JOIN dim_cliente c
    ON f.cliente_sk = c.cliente_sk
WHERE c.customer_state = 'SP'
GROUP BY categoria
ORDER BY receita_total DESC;
""").df()

,categoria,qtd_pedidos,receita_total
0,cama_mesa_banho,4347,549061.89
1,beleza_saude,3715,510255.38
2,relogios_presentes,2090,448582.76
3,esporte_lazer,3216,428212.45
4,informatica_acessorios,2616,386280.65
...,...,...,...
68,casa_conforto_2,11,624.56
69,pc_gamer,2,595.53
70,flores,10,558.12
71,cds_dvds_musicais,6,409.64


## Exercício 8 🟠 — HAVING com pedidos por estado

Liste apenas os estados com **mais de 500 pedidos únicos**.

A consulta deve retornar:

- `estado`
- `qtd_pedidos`
- `receita_total`

Ordene do maior para o menor número de pedidos.

**Objetivo:** praticar filtro de grupos com `HAVING`.

In [12]:
banco_dados.sql("""
SELECT
    c.customer_state AS estado,
    COUNT(DISTINCT f.order_id) as qtd_pedidos,
    SUM(f.valor_total) AS receita_total,
FROM fato_vendas f
INNER JOIN dim_cliente c
    ON f.cliente_sk = c.cliente_sk
GROUP BY estado
HAVING qtd_pedidos > 500
ORDER BY qtd_pedidos DESC;    
""").df()

,estado,qtd_pedidos,receita_total
0,SP,40501,5769703.15
1,RJ,12350,2055401.57
2,MG,11354,1818891.67
3,RS,5345,861472.79
4,PR,4923,781708.80
5,SC,3546,595127.78
6,BA,3256,591137.81
7,DF,2080,346123.35
8,ES,1995,317657.93
9,GO,1957,334212.35


## Exercício 9 🟠 — Receita mensal com DATE_TRUNC

Calcule o **faturamento por mês** usando `DATE_TRUNC`.

Use `fato_vendas` e `dim_tempo`.

A consulta deve retornar:

- `mes_referencia`, usando `DATE_TRUNC('month', CAST(t.data AS DATE))`
- `qtd_pedidos`
- `receita_total`

Ordene pelo mês em ordem crescente.

**Objetivo:** praticar agregação temporal com `DATE_TRUNC`.

In [13]:
banco_dados.sql("""
SELECT
    DATE_TRUNC('month', CAST(t.data AS DATE)) AS mes_referencia,
    COUNT(DISTINCT f.order_id) as qtd_pedidos,
    SUM(f.valor_total) AS receita_total
FROM fato_vendas f
INNER JOIN dim_tempo t
    ON f.tempo_sk = t.tempo_sk
GROUP BY mes_referencia
ORDER BY mes_referencia ASC;
""").df()

,mes_referencia,qtd_pedidos,receita_total
0,2016-09-01,1,143.46
1,2016-10-01,265,46490.66
2,2016-12-01,1,19.62
3,2017-01-01,750,127482.37
4,2017-02-01,1653,271239.32
5,2017-03-01,2546,414330.95
6,2017-04-01,2303,390812.40
7,2017-05-01,3546,566851.40
8,2017-06-01,3135,490050.37
9,2017-07-01,3872,566299.08


## Exercício 10 🔴 — Desafio BI: sazonalidade por trimestre e categoria

Monte uma análise de **receita por trimestre e categoria**.

Use `fato_vendas`, `dim_produto` e `dim_tempo`.

A consulta deve retornar:

- `ano`, usando `EXTRACT(YEAR FROM CAST(t.data AS DATE))`
- `trimestre`, usando `EXTRACT(QUARTER FROM CAST(t.data AS DATE))`
- `categoria`
- `qtd_pedidos`
- `receita_total`
- `ticket_medio`

Mostre apenas grupos com `receita_total` acima de **50000**.

Ordene por:

1. `ano`
2. `trimestre`
3. `receita_total` do maior para o menor

**Objetivo:** combinar `JOIN`, `GROUP BY`, `EXTRACT`, `HAVING` e `ORDER BY`.

In [14]:
banco_dados.sql("""
SELECT 
    EXTRACT(YEAR FROM CAST(t.data AS DATE)) AS ano,
    EXTRACT(QUARTER FROM CAST(t.data AS DATE)) AS trimestre,
    p.product_category_name AS categoria,
    COUNT(f.order_id) AS qtd_pedidos,
    SUM(f.valor_total) AS receita_total,
    AVG(f.valor_total) AS ticket_medio
from fato_vendas f
INNER JOIN dim_produto p
    ON f.produto_sk = p.produto_sk
INNER JOIN dim_tempo t
    ON f.tempo_sk = t.tempo_sk
GROUP BY 
    EXTRACT(YEAR FROM CAST(t.data AS DATE)),
    EXTRACT(QUARTER FROM CAST(t.data AS DATE)),
    categoria
HAVING receita_total > 50000
ORDER BY ano, trimestre, receita_total DESC;
""").df()

,ano,trimestre,categoria,qtd_pedidos,receita_total,ticket_medio
0,2017,1,moveis_decoracao,749,68664.51,91.674913
1,2017,1,beleza_saude,444,68445.11,154.155653
2,2017,1,esporte_lazer,413,58618.11,141.932470
3,2017,1,cama_mesa_banho,482,52956.32,109.867884
4,2017,2,beleza_saude,718,114553.78,159.545655
...,...,...,...,...,...,...
91,2018,3,telefonia,492,71894.99,146.128028
92,2018,3,bebes,426,52849.88,124.060751
93,2018,3,perfumaria,417,52229.59,125.250815
94,2018,3,pet_shop,356,51492.38,144.641517


## Checklist final

Antes de entregar, confira:

- Todas as consultas executaram sem erro.
- As consultas com `JOIN` usam as chaves corretas: `cliente_sk`, `produto_sk` e `tempo_sk`.
- As consultas agregadas usam `GROUP BY` para todas as colunas não agregadas.
- Filtros antes da agregação usam `WHERE`.
- Filtros depois da agregação usam `HAVING`.
- Os resultados estão ordenados conforme o enunciado.
